# Meta DE Screen — 5 SQL Problems

The 60-minute CoderPad SQL half. The 5 problems below are the 2026 Meta DE screen format. Each is a 5-minute target.

**Pass bar:** 3 of 5. Code runs against a real test-case suite.

Pairs with `design/02_sql_problems.md` for the worked solutions + the 4 things each problem tests.

**Note on Problem 1.** The worked solution filters countries with `total >= 1000` users (real Meta scale). The sample data has only 4 users per country, so Problem 1 returns an empty result set *by design* — that's the right answer at Meta scale. The `tests/test_meta_screen_sql.py` asserts the empty result. To see the cohort math non-empty, drop the threshold from `1000` to `1` in the worked solution.

In [1]:
import os, re, sqlite3
BASE = os.path.abspath(os.path.join(os.getcwd(), '..')) if 'notebooks' in os.getcwd() else os.getcwd()
schema_path = os.path.join(BASE, 'code', 'meta_schema.sql')
sol_path = os.path.join(BASE, 'code', 'meta_screen_sql.sql')

conn = sqlite3.connect(':memory:')
with open(schema_path) as f:
    conn.executescript(f.read())

def _load(n):
    txt = open(sol_path).read()
    m = re.search(rf'^--\s*Problem\s+{n}:[^\n]*\n(.*?)(?=^--\s*Problem\s+\d+:|\Z)',
                  txt, re.DOTALL | re.MULTILINE)
    body = m.group(1)
    return '\n'.join(l for l in body.splitlines()
                     if l.strip() and not l.strip().startswith('--')).strip().rstrip(';').strip()
print('schema + 5 problems loaded')

schema + 5 problems loaded


## Problem 1 — 7-day rolling retention by country

Window function + cohort. Filter *after* aggregation, not before. The sample data is below the 1000-user threshold Meta uses in prod, so the result is empty by design.

In [2]:
rows = list(conn.execute(_load(1)))
print(f'rows returned: {len(rows)} (expected 0 at sample-data scale)')
for row in rows:
    print(row)

rows returned: 0 (expected 0 at sample-data scale)


## Problem 2 — Peak engagement in first hour

JOIN on a time-window predicate, then compare each post's hour-1 engagement to its page's peak hour-1.

In [3]:
for row in conn.execute(_load(2)):
    print(row)

(1, 301, 0, 0)
(2, 301, 0, 0)
(3, 301, 0, 0)
(4, 302, 0, 0)
(5, 302, 0, 0)


## Problem 3 — Top-3 posts per qualified page

Qualifying-page filter (HAVING MIN >= 100) before the ROW_NUMBER() partition. The HAVING clause is what makes this a non-trivial interview problem.

In [4]:
for row in conn.execute(_load(3)):
    print(row)

(2, 301, 1500)
(1, 301, 1000)
(3, 301, 800)
(5, 302, 600)
(4, 302, 500)


## Problem 4 — Sessionize events (30-min gap)

The signature Meta pattern. `LAG` for the previous event + cumulative-sum flag for the session_id.

In [5]:
for row in conn.execute(_load(4)):
    print(row)

(201, 1, '2026-01-01T08:00:00', '2026-01-01T08:05:00', 2)
(201, 2, '2026-01-01T08:45:00', '2026-01-01T08:45:00', 1)
(202, 1, '2026-01-01T09:00:00', '2026-01-01T09:00:00', 1)
(202, 2, '2026-01-02T09:00:00', '2026-01-02T09:00:00', 1)
(203, 1, '2026-01-01T10:00:00', '2026-01-01T10:00:00', 1)
(203, 2, '2026-01-02T10:00:00', '2026-01-02T10:00:00', 1)
(203, 3, '2026-01-03T10:00:00', '2026-01-03T10:00:00', 1)
(203, 4, '2026-01-04T10:00:00', '2026-01-04T10:00:00', 1)
(204, 1, '2026-01-01T12:00:00', '2026-01-01T12:00:00', 1)


## Problem 5 — Gaps-and-islands (longest streak)

Row-number date trick: a consecutive sequence has a constant `DATE(d, '-' || rn || ' days')` value across rows.

In [6]:
for row in conn.execute(_load(5)):
    print(row)

(203, 4)
